# Fraud Detection - Exploratory Data Analysis

This notebook explores the transaction dataset and helps understand patterns in fraudulent vs legitimate transactions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load raw data
data_path = Path('../data/raw/transactions.csv')

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df):,} transactions")
else:
    print("Data not found. Run: python scripts/download_data.py")

In [ ]:
# Display first few rows
df.head()

## 2. Data Overview

In [ ]:
# Dataset info
print("Dataset Shape:", df.shape)
print("\nColumn Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()

## 3. Fraud Distribution

In [ ]:
# Fraud rate
fraud_rate = df['is_fraud'].mean()
print(f"Fraud Rate: {fraud_rate:.2%}")
print(f"Fraud Cases: {df['is_fraud'].sum():,}")
print(f"Legitimate Cases: {(~df['is_fraud'].astype(bool)).sum():,}")

# Visualize
plt.figure(figsize=(8, 6))
df['is_fraud'].value_counts().plot(kind='bar')
plt.title('Transaction Distribution')
plt.xlabel('Is Fraud')
plt.ylabel('Count')
plt.xticks([0, 1], ['Legitimate', 'Fraud'], rotation=0)
plt.show()

## 4. Amount Analysis

In [ ]:
# Amount distribution by fraud status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df[df['is_fraud']==0]['amount'], bins=50, alpha=0.6, label='Legitimate')
axes[0].hist(df[df['is_fraud']==1]['amount'], bins=50, alpha=0.6, label='Fraud')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Amount Distribution')
axes[0].legend()

# Box plot
df.boxplot(column='amount', by='is_fraud', ax=axes[1])
axes[1].set_xlabel('Is Fraud')
axes[1].set_ylabel('Amount')
axes[1].set_title('Amount by Fraud Status')

plt.tight_layout()
plt.show()

# Statistics
print("\nAmount Statistics by Fraud Status:")
print(df.groupby('is_fraud')['amount'].describe())

## 5. Temporal Patterns

In [ ]:
# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Fraud by hour
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hour of day
fraud_by_hour = df.groupby(['hour', 'is_fraud']).size().unstack(fill_value=0)
fraud_by_hour.plot(kind='bar', ax=axes[0], alpha=0.7)
axes[0].set_title('Transactions by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')
axes[0].legend(['Legitimate', 'Fraud'])

# Day of week
fraud_by_day = df.groupby(['day_of_week', 'is_fraud']).size().unstack(fill_value=0)
fraud_by_day.plot(kind='bar', ax=axes[1], alpha=0.7)
axes[1].set_title('Transactions by Day of Week')
axes[1].set_xlabel('Day (0=Monday)')
axes[1].set_ylabel('Count')
axes[1].legend(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

## 6. Merchant Analysis

In [ ]:
# Fraud by merchant category
if 'merchant_category' in df.columns:
    fraud_by_category = df.groupby('merchant_category')['is_fraud'].agg(['sum', 'count', 'mean'])
    fraud_by_category.columns = ['Fraud_Count', 'Total_Count', 'Fraud_Rate']
    fraud_by_category = fraud_by_category.sort_values('Fraud_Rate', ascending=False)
    
    print("Fraud Rate by Merchant Category:")
    print(fraud_by_category)
    
    # Visualize
    fraud_by_category['Fraud_Rate'].plot(kind='bar', figsize=(10, 6))
    plt.title('Fraud Rate by Merchant Category')
    plt.xlabel('Category')
    plt.ylabel('Fraud Rate')
    plt.xticks(rotation=45)
    plt.show()

## 7. Correlation Analysis

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

# Correlation with fraud
fraud_corr = df[numeric_cols].corr()['is_fraud'].sort_values(ascending=False)
print("\nCorrelation with Fraud:")
print(fraud_corr)

## 8. Key Insights

Based on the analysis above, document key findings:

1. **Fraud Rate**: [Fill in]
2. **Amount Patterns**: Fraudulent transactions tend to...
3. **Temporal Patterns**: Fraud is more common during...
4. **High-Risk Categories**: [...]
5. **Feature Engineering Ideas**: [...]

## Next Steps

1. Run ETL pipeline: `python etl/bronze_layer.py`
2. Engineer features: `python etl/silver_layer.py`
3. Create feature store: `python etl/gold_layer.py`
4. Train model: `python ml/train.py`